# ==========================================================
# Enterprise AI Fraud Detection Platform
# Notebook : 01_Data_Audit.ipynb
# Phase    : Phase 1 — Enterprise Dataset Intelligence Audit
# Author   : Somsubhra Dalui
# Version  : v1.0
# Objective:
# Understand the dataset before preprocessing or modeling.
# ==========================================================

> **Philosophy:** When a bank receives a new dataset they don't ask *"Which model should we use?"*
> They ask: *Is the data reliable? Is there leakage? Are there hidden biases? Can this be deployed?*
>
> That is exactly what this notebook answers.

## Section 1 — Import Libraries

Only import what we need. No model-related packages at this stage.

In [1]:
from src.utils.bootstrap import bootstrap
bootstrap()

import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")            # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Ensure output directories exist
os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../reports/tables", exist_ok=True)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Section 2 — Load Dataset

We always start from the **raw, immutable** data. We never modify this file directly.
Every transformation happens downstream in cleaned copies.

In [2]:
DATA_PATH = "../data/raw/bank_fraud_dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
df.head()

Dataset loaded: 9,082 rows × 3,925 columns


Unnamed: 0  F1  F2  F3  F4  F5  F6  F7  F8  F9  F10  F11  F12   F13   F14  \
0           1 NaN NaN NaN NaN NaN NaN NaN NaN NaN  NaN  NaN  NaN  0.59  0.74   
1           2 NaN NaN NaN NaN NaN NaN NaN NaN NaN  NaN  NaN  NaN  0.80  0.86   
2           3 NaN NaN NaN NaN NaN NaN NaN NaN NaN  NaN  NaN  NaN  0.76  0.88   
3           4 NaN NaN NaN NaN NaN NaN NaN NaN NaN  NaN  NaN  NaN  0.46  0.54   
4           5 NaN NaN NaN NaN NaN NaN NaN NaN NaN  NaN  NaN  NaN  0.50  0.64   

    F15   F16   F17   F18   F19   F20   F21   F22   F23   F24   F25   F26  \
0  0.38  0.64  0.67  0.63  0.59  0.74  0.38  0.64  0.67  0.63  0.59  0.74   
1  0.79  0.95  0.94  0.96  0.80  0.86  0.79  0.95  0.94  0.96  0.80  0.86   
2  0.67  0.95  0.93  0.98  0.76  0.88  0.67  0.95  0.93  0.98  0.76  0.88   
3  0.46  0.94  0.96  0.92  0.46  0.54  0.46  0.94  0.96  0.92  0.46  0.54   
4  0.46  0.39  0.43  0.35  0.50  0.64  0.46  0.39  0.43  0.35  0.50  0.64   

    F27   F28   F29   F30  F31  F32  F33  F34  F35  F36  F37  F38  F39  F40  \
0  0.38  0.64  0.67  0.63  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
1  0.79  0.95  0.94  0.96  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2  0.67  0.95  0.93  0.98  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
3  0.46  0.94  0.96  0.92  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
4  0.46  0.39  0.43  0.35  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

   F41  F42  F43  F44  F45  F46  F47  F48  F49  F50  F51  F52   F53   F54  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  0.00  0.00   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  0.80  0.80   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1.00  1.00   
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  0.86  0.86   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  0.90  0.90   

   F55   F56   F57  F58   F59   F60   F61   F62   F63   F64  F65  F66  F67  \
0  NaN  0.00  0.00  NaN  0.59  0.79  0.38  0.65  0.74  0.63  0.0  0.0  NaN   
1  NaN  0.94  0.94  NaN  0.80  1.00  0.79  0.96  1.00  0.96  NaN  NaN  NaN   
2  NaN  1.00  1.00  NaN  0.76  0.88  0.67  0.95  0.93  0.98  NaN  NaN  NaN   
3  NaN  0.98  0.98  NaN  0.44  0.00  0.45  0.71  0.00  0.78  1.0  1.0  NaN   
4  NaN  0.99  0.99  NaN  0.47  0.50  0.46  0.35  0.35  0.35  NaN  NaN  NaN   

   F68  F69  F70  F71  F72  F73  F74  F75  F76  F77  F78  F79  F80  F81  F82  \
0  0.0  0.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
3  1.0  1.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

   F83  F84  F85  F86  F87  F88  F89  F90  F91  F92  F93  F94  F95  F96  F97  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

   F98  F99  F100  F101  F102  F103  F104  F105  F106  F107  F108  F109  F110  \
0  NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.53  0.56   
1  NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.30  0.21   
2  NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.42  0.49   
3  NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.69  0.54   
4  NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.53  0.51   

   F111  F112  F113  F114  F115  F116  F117  F118  F119  F120  F121  F122  \
0  0.50  0.51  0.43  0.56  0.53  0.56  0.50  0.51  0.43  0.56  0.53  0.56   
1  0.33  0.17  0.15  0.19  0.30  0.21  0.33  0.17

## Section 3 — Basic Dataset Information

We inspect schema, data types, memory footprint, and surface-level statistics.

**What we are looking for:**
- Numerical vs categorical columns
- Memory usage
- Missing values at a glance
- Feature types and ranges

In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 3925 columns):
 #     Column      Non-Null Count  Dtype  
---    ------      --------------  -----  
 0     Unnamed: 0  9082 non-null   int64  
 1     F1          2591 non-null   float64
 2     F2          1532 non-null   float64
 3     F3          1448 non-null   float64
 4     F4          2591 non-null   float64
 5     F5          1532 non-null   float64
 6     F6          1448 non-null   float64
 7     F7          855 non-null    float64
 8     F8          488 non-null    float64
 9     F9          725 non-null    float64
 10    F10         855 non-null    float64
 11    F11         488 non-null    float64
 12    F12         725 non-null    float64
 13    F13         8997 non-null   float64
 14    F14         8690 non-null   float64
 15    F15         8748 non-null   float64
 16    F16         8997 non-null   float64
 17    F17         8690 non-null   float64
 18    F18         8748 non-nul

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,9082.0,4541.500000,2621.891906,1.0,2271.25,4541.5,6811.75,9082.0
F1,2591.0,0.679722,0.392573,0.0,0.43,1.0,1.00,1.0
F2,1532.0,0.656129,0.398420,0.0,0.33,1.0,1.00,1.0
F3,1448.0,0.724731,0.408869,0.0,0.50,1.0,1.00,1.0
F4,2591.0,0.691721,0.401105,0.0,0.38,1.0,1.00,1.0
...,...,...,...,...,...,...,...,...
F3920,9082.0,0.210636,0.569385,0.0,0.00,0.0,0.00,14.0
F3921,9082.0,0.336930,0.647817,0.0,0.00,0.0,1.00,11.0
F3922,9082.0,0.562431,0.761297,0.0,0.00,0.0,1.00,18.0
F3923,9082.0,0.066505,0.325110,0.0,0.00,0.0,0.00,6.0


In [5]:
df.describe(include="object").T

,count,unique,top,freq
F2230,9082,4,Oct25,9001
F3886,9082,17,Savings,5956
F3888,9082,4292,9-19-2025,24
F3889,9082,7,G365D,7544
F3890,9082,4,M,2900
F3891,9082,7,selfemployed,3951
F3892,6484,3,M,5007
F3893,9082,2,RETAIL,6437


## Section 4 — Dataset Overview

| Metric | Value |
|--------|-------|
| **Observations** | 9,082 |
| **Features** | 3,925 |
| **Target variable** | `F3924` (binary: 0 = legitimate, 1 = fraud) |
| **float64 features** | 3,876 |
| **int64 features** | 41 |
| **Categorical / string features** | 8 |
| **Memory usage** | ~288 MB |

> This is a **wide** dataset — almost 4,000 features for fewer than 10,000 observations.
> High dimensionality + low sample size is a classic challenge in financial ML.
> Feature selection will be critical in later phases.

## Section 5 — Target Analysis

The target variable `F3924` indicates whether a transaction/account is fraudulent.
We must understand the class distribution before any modeling decision.

In [6]:
print("Absolute counts:")
print(df["F3924"].value_counts())
print()
print("Percentage distribution:")
print(df["F3924"].value_counts(normalize=True) * 100)

Absolute counts:
F3924
0    9001
1      81
Name: count, dtype: int64

Percentage distribution:
F3924
0    99.108126
1     0.891874
Name: proportion, dtype: float64


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2ecc71", "#e74c3c"]
labels = ["Non-Fraud (0)", "Fraud (1)"]

# --- Bar chart ---
counts = df["F3924"].value_counts().sort_index()
axes[0].bar(labels, counts.values, color=colors, edgecolor="black", linewidth=0.8)
axes[0].set_title("Target Distribution (Count)", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count", fontsize=12)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 80, f"{v:,}", ha="center", fontweight="bold", fontsize=13)

# --- Pie chart ---
axes[1].pie(counts.values, labels=labels, colors=colors, autopct="%1.2f%%",
            startangle=90, explode=(0, 0.12), shadow=True,
            textprops={"fontsize": 12}, pctdistance=0.55)
axes[1].set_title("Target Distribution (Proportion)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: reports/figures/target_distribution.png")

✅ Saved: reports/figures/target_distribution.png

### Interpretation — Severe Class Imbalance

| Class | Count | Percentage |
|-------|-------|-----------|
| Non-Fraud (0) | 9,001 | 99.11 % |
| Fraud (1) | 81 | **0.89 %** |

**Imbalance ratio ≈ 111 : 1**

This extreme imbalance means:
- A naïve model that predicts "non-fraud" for every record achieves **99.11 % accuracy** — yet catches zero fraud.
- We **must** use class-aware metrics: **AUPRC**, **F1-score**, **recall at fixed precision**.
- Resampling strategies (SMOTE, ADASYN) or cost-sensitive learning will be essential in the modelling phase.

## Section 6 — Duplicate Analysis

Duplicate records can inflate model performance and bias evaluation.
In banking, the same customer may appear multiple times — we need to verify.

In [8]:
full_dupes = df.duplicated().sum()
excl_target = df.duplicated(subset=df.columns.drop("F3924")).sum()

print(f"Full duplicate rows:              {full_dupes}")
print(f"Duplicates (excluding target):    {excl_target}")

Full duplicate rows:              0
Duplicates (excluding target):    0


### Result
- **Zero full duplicates** — every row is unique.
- **Zero duplicates when excluding the target** — no record appears as both fraud and non-fraud.

This is a clean result. No deduplication needed.

## Section 7 — Missing Value Audit

Missing data is one of the most common data-quality issues in financial datasets.
Features with excessive missingness carry little signal and add noise.

**Thresholds we will use:**
- **100 % missing** → fully empty → immediate removal
- **> 95 % missing** → excessive sparsity → strong candidate for removal
- **> 50 % missing** → investigate further
- **0 % missing** → fully populated → highest quality

In [9]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_sorted = missing_percent[missing_percent > 0].sort_values(ascending=False)

print(f"Columns with 100% missing (fully empty): {(missing_percent == 100).sum()}")
print(f"Columns with > 95% missing:              {(missing_percent > 95).sum()}")
print(f"Columns with > 50% missing:              {(missing_percent > 50).sum()}")
print(f"Columns with > 0% missing:               {(missing_percent > 0).sum()}")
print(f"Columns with 0% missing (complete):      {(missing_percent == 0).sum()}")
print(f"\nTotal columns: {len(missing_percent)}")

Columns with 100% missing (fully empty): 63
Columns with > 95% missing:              361
Columns with > 50% missing:              1138
Columns with > 0% missing:               3835
Columns with 0% missing (complete):      90

Total columns: 3925


In [10]:
print("Top 30 columns by missing percentage:\n")
for rank, (col, pct) in enumerate(missing_sorted.head(30).items(), 1):
    print(f"  {rank:>2}. {col:<10}  {pct:>6.2f} %")

Top 30 columns by missing percentage:

   1. F3776       100.00 %
   2. F3773       100.00 %
   3. F3668       100.00 %
   4. F3665       100.00 %
   5. F542        100.00 %
   6. F2655       100.00 %
   7. F539        100.00 %
   8. F440        100.00 %
   9. F2312       100.00 %
  10. F437        100.00 %
  11. F2756       100.00 %
  12. F492        100.00 %
  13. F495        100.00 %
  14. F2607       100.00 %
  15. F3133       100.00 %
  16. F3182       100.00 %
  17. F3179       100.00 %
  18. F3236       100.00 %
  19. F3233       100.00 %
  20. F2707       100.00 %
  21. F185        100.00 %
  22. F189        100.00 %
  23. F192        100.00 %
  24. F2971       100.00 %
  25. F2921       100.00 %
  26. F2917       100.00 %
  27. F3344       100.00 %
  28. F3341       100.00 %
  29. F2458       100.00 %
  30. F239        100.00 %


In [11]:
fig, ax = plt.subplots(figsize=(14, 8))
missing_sorted.head(30).plot(kind="barh", ax=ax, color="#e74c3c", edgecolor="black", linewidth=0.5)
ax.set_title("Top 30 Columns by Missing Value Percentage", fontsize=14, fontweight="bold")
ax.set_xlabel("Missing %", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
ax.axvline(x=95, color="red", linestyle="--", linewidth=1.5, label="95% threshold")
ax.axvline(x=50, color="orange", linestyle="--", linewidth=1.5, label="50% threshold")
ax.legend(fontsize=11, loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../reports/figures/missing_values_top30.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: reports/figures/missing_values_top30.png")

✅ Saved: reports/figures/missing_values_top30.png


In [12]:
fig, ax = plt.subplots(figsize=(12, 5))
missing_percent[missing_percent > 0].hist(bins=50, ax=ax, color="#3498db", edgecolor="black")
ax.set_title("Distribution of Missing Value Percentages Across Features", fontsize=14, fontweight="bold")
ax.set_xlabel("Missing %", fontsize=12)
ax.set_ylabel("Number of Features", fontsize=12)
ax.axvline(x=95, color="red", linestyle="--", linewidth=1.5, label="95% threshold")
ax.axvline(x=50, color="orange", linestyle="--", linewidth=1.5, label="50% threshold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("../reports/figures/missing_value_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: reports/figures/missing_value_distribution.png")

✅ Saved: reports/figures/missing_value_distribution.png


In [13]:
try:
    import missingno as msno

    # Select top 30 columns with most missing data for the matrix
    sample_cols = list(missing_sorted.head(30).index)
    ax = msno.matrix(df[sample_cols], figsize=(16, 8), fontsize=9, sparkline=False)
    plt.title("Missing Value Pattern — Top 30 Missing Columns", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../reports/figures/missingno_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved: reports/figures/missingno_matrix.png")
except ImportError:
    print("⚠️  missingno not installed — skipping matrix plot")

✅ Saved: reports/figures/missingno_matrix.png


### Missing Value Summary

| Category | Count | Action |
|----------|-------|--------|
| Fully empty (100 %) | **63** | Remove immediately |
| > 95 % missing | **361** | Remove — too sparse for reliable modelling |
| > 50 % missing | **1,138** | Investigate — may contain recoverable signal |
| 0 % missing | **90** | Highest quality features |

> **Key insight:** Over a third of all features (1,138 / 3,925 = **29 %**) have more missing
> values than actual data. This extreme sparsity pattern suggests many features were only
> computed for specific account types or time windows.

## Section 8 — Data Type Audit

Separate numeric, categorical, and potential datetime features for targeted analysis.

In [14]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print(f"Numeric features:     {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Total:                {len(numeric_cols) + len(categorical_cols)}")
print()
print("Categorical columns:")
for col in categorical_cols:
    print(f"  • {col}  (dtype={df[col].dtype}, unique={df[col].nunique()})")
print()
print("Potential datetime column:")
print(f"  • F3888  — sample values: {df['F3888'].head(3).tolist()}")
print(f"    Date range after parsing: ", end="")
dates = pd.to_datetime(df["F3888"], format="mixed", errors="coerce")
print(f"{dates.min()} → {dates.max()}")

Numeric features:     3917
Categorical features: 8
Total:                3925

Categorical columns:
  • F2230  (dtype=object, unique=4)
  • F3886  (dtype=object, unique=17)
  • F3888  (dtype=object, unique=4292)
  • F3889  (dtype=object, unique=7)
  • F3890  (dtype=object, unique=4)
  • F3891  (dtype=object, unique=7)
  • F3892  (dtype=object, unique=3)
  • F3893  (dtype=object, unique=2)

Potential datetime column:
  • F3888  — sample values: ['8-1-2011', '8-2-2011', '8-4-2011']
    Date range after parsing: 1900-01-03 00:00:00 → 2025-11-28 00:00:00


### Data Type Summary

| Type | Count | Notes |
|------|-------|-------|
| **Numeric (float64)** | 3,876 | Engineered behavioural features |
| **Numeric (int64)** | 41 | Counts, flags, identifiers |
| **Categorical (string)** | 8 | Business metadata |
| **Datetime (to parse)** | 1 | `F3888` — account opening date |

> `F3888` is currently stored as a string. It should be parsed to datetime in Phase 2
> to extract temporal features (account age, day-of-week, month, etc.).

## Section 9 — Categorical Feature Audit

For every categorical column we examine:
- Number of unique values (cardinality)
- Missing values
- Top categories and their frequencies

**We do NOT encode anything yet** — that belongs to Phase 2/3.

In [15]:
for col in categorical_cols:
    print("=" * 65)
    print(f"Column: {col}")
    print(f"  Dtype:          {df[col].dtype}")
    print(f"  Unique values:  {df[col].nunique()}")
    miss = df[col].isnull().sum()
    print(f"  Missing values: {miss} ({miss / len(df) * 100:.2f}%)")
    print(f"  Top categories:")
    for val, cnt in df[col].value_counts().head(5).items():
        pct = cnt / len(df) * 100
        bar = "█" * int(pct / 2)
        print(f"    {val:>20s} : {cnt:>5,}  ({pct:5.2f}%)  {bar}")
    print()

Column: F2230
  Dtype:          object
  Unique values:  4
  Missing values: 0 (0.00%)
  Top categories:
                   Oct25 : 9,001  (99.11%)  █████████████████████████████████████████████████
                   Sep25 :    48  ( 0.53%)  
                   Nov25 :    23  ( 0.25%)  
                   Dec25 :    10  ( 0.11%)  

Column: F3886
  Dtype:          object
  Unique values:  17
  Missing values: 0 (0.00%)
  Top categories:
                 Savings : 5,956  (65.58%)  ████████████████████████████████
                 Current : 2,051  (22.58%)  ███████████
              MSME Micro :   337  ( 3.71%)  █
              MSME Small :   242  ( 2.66%)  █
             Staff Loans :   108  ( 1.19%)  

Column: F3888
  Dtype:          object
  Unique values:  4292
  Missing values: 0 (0.00%)
  Top categories:
               9-19-2025 :    24  ( 0.26%)  
               9-17-2025 :    20  ( 0.22%)  
              10-13-2025 :    18  ( 0.20%)  
               9-23-2025 :    16  ( 0.18%)  


## Section 10 — Numerical Feature Audit

Financial datasets are usually **highly skewed**. We compute:
- Standard descriptive statistics (mean, median, std, min, max)
- Skewness across all numeric features

This informs whether we need log transforms, robust scaling, or outlier treatment.

In [16]:
print(f"Number of numeric features: {len(numeric_cols)}")
print()
print("Descriptive statistics (first 20 numeric features):")
df[numeric_cols[:20]].describe().T

Number of numeric features: 3917

Descriptive statistics (first 20 numeric features):


,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,9082.0,4541.500000,2621.891906,1.0,2271.25,4541.500,6811.75,9082.0
F1,2591.0,0.679722,0.392573,0.0,0.43,1.000,1.00,1.0
F2,1532.0,0.656129,0.398420,0.0,0.33,1.000,1.00,1.0
F3,1448.0,0.724731,0.408869,0.0,0.50,1.000,1.00,1.0
F4,2591.0,0.691721,0.401105,0.0,0.38,1.000,1.00,1.0
F5,1532.0,0.659497,0.404242,0.0,0.32,1.000,1.00,1.0
F6,1448.0,0.727811,0.413717,0.0,0.43,1.000,1.00,1.0
F7,855.0,0.487135,0.399929,0.0,0.00,0.500,1.00,1.0
F8,488.0,0.441844,0.435882,0.0,0.00,0.330,1.00,1.0
F9,725.0,0.508676,0.398278,0.0,0.00,0.500,1.00,1.0


In [17]:
skewness = df[numeric_cols].skew().abs().sort_values(ascending=False)

print("Top 20 most skewed features (absolute skewness):\n")
for rank, (col, val) in enumerate(skewness.head(20).items(), 1):
    print(f"  {rank:>2}. {col:<10}  |skew| = {val:.4f}")

print(f"\nFeatures with |skew| > 2:   {(skewness > 2).sum()}")
print(f"Features with |skew| > 10:  {(skewness > 10).sum()}")
print(f"Features with |skew| > 50:  {(skewness > 50).sum()}")

Top 20 most skewed features (absolute skewness):

   1. F890        |skew| = 95.2890
   2. F1245       |skew| = 95.2890
   3. F1569       |skew| = 95.2890
   4. F865        |skew| = 95.2890
   5. F867        |skew| = 95.2890
   6. F835        |skew| = 95.2890
   7. F832        |skew| = 95.2890
   8. F917        |skew| = 95.2890
   9. F916        |skew| = 95.2890
  10. F919        |skew| = 95.2890
  11. F920        |skew| = 95.2890
  12. F864        |skew| = 95.2890
  13. F862        |skew| = 95.2890
  14. F887        |skew| = 95.2890
  15. F901        |skew| = 95.2890
  16. F1566       |skew| = 95.2890
  17. F1242       |skew| = 95.2890
  18. F898        |skew| = 95.2890
  19. F812        |skew| = 95.2470
  20. F811        |skew| = 95.2470

Features with |skew| > 2:   2838
Features with |skew| > 10:  2006
Features with |skew| > 50:  611


In [18]:
fig, ax = plt.subplots(figsize=(12, 5))
skewness.clip(upper=100).hist(bins=60, ax=ax, color="#9b59b6", edgecolor="black")
ax.set_title("Distribution of Feature Skewness (|skew|, clipped at 100)", fontsize=14, fontweight="bold")
ax.set_xlabel("|Skewness|", fontsize=12)
ax.set_ylabel("Number of Features", fontsize=12)
ax.axvline(x=2, color="red", linestyle="--", linewidth=1.5, label="|skew| = 2 (high)")
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("../reports/figures/skewness_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: reports/figures/skewness_distribution.png")

✅ Saved: reports/figures/skewness_distribution.png


## Section 11 — Leakage Investigation 🔍

> **Target leakage** is the single most dangerous flaw in any ML pipeline.
> A model trained with leaked features will appear to perform brilliantly in development
> but will **fail catastrophically in production** — where the leaked information is unavailable.
>
> In AML (Anti-Money Laundering) systems, deploying a model with leakage can mean
> regulatory fines, missed fraud, and loss of banking license.

We investigate two suspicious columns identified during initial profiling.

### 11.1 — `Unnamed: 0` Investigation

**Hypothesis:** This column is a row-index artifact from a previous CSV export.

In [19]:
print("=" * 60)
print("🔍 FORENSIC INVESTIGATION: Unnamed: 0")
print("=" * 60)

col = "Unnamed: 0"
print(f"\n  Present in dataset:        {col in df.columns}")
print(f"  Min value:                 {df[col].min()}")
print(f"  Max value:                 {df[col].max()}")
print(f"  Unique values:             {df[col].nunique()}")
print(f"  Equals row count:          {df[col].nunique() == len(df)}")
print(f"  Monotonically increasing:  {df[col].is_monotonic_increasing}")
print(f"  Correlation with F3924:    {df[col].corr(df['F3924']):.4f}")

print()
print("📋 EVIDENCE:")
print("  1. Values run from 1 to 9,082 — exactly matching the row count.")
print("  2. Monotonically increasing — classic index signature.")
print("  3. Correlation of 0.16 with the target is non-trivial.")
print("  4. If fraud cases are clustered at certain index positions,")
print("     a model could learn this spurious positional pattern.")
print()
print("⚖️  VERDICT:  Row-index artifact with positional leakage risk.")
print("🚨 DECISION: REMOVE in Phase 2.")

🔍 FORENSIC INVESTIGATION: Unnamed: 0

  Present in dataset:        True
  Min value:                 1
  Max value:                 9082
  Unique values:             9082
  Equals row count:          True
  Monotonically increasing:  True
  Correlation with F3924:    0.1628

📋 EVIDENCE:
  1. Values run from 1 to 9,082 — exactly matching the row count.
  2. Monotonically increasing — classic index signature.
  3. Correlation of 0.16 with the target is non-trivial.
  4. If fraud cases are clustered at certain index positions,
     a model could learn this spurious positional pattern.

⚖️  VERDICT:  Row-index artifact with positional leakage risk.
🚨 DECISION: REMOVE in Phase 2.


#### Forensic Summary — `Unnamed: 0`

| Evidence | Finding |
|----------|---------|
| Value range | 1 → 9,082 (matches row count exactly) |
| Monotonicity | Strictly increasing |
| Correlation with target | 0.1628 |
| Business meaning | **None** — artefact of `pd.to_csv()` without `index=False` |

**Risk:** A tree-based model could learn to split on index ranges that happen to
concentrate fraud cases (e.g., "if row > 8500 then fraud"). This would not generalise.

**Decision:** ❌ **Remove** — no business meaning, positional leakage risk.

### 11.2 — `F3912` Investigation

**Hypothesis:** This column is a copy or near-copy of the target variable `F3924`.

In [20]:
print("=" * 60)
print("🔍 FORENSIC INVESTIGATION: F3912")
print("=" * 60)

print(f"\n  Unique values: {df['F3912'].nunique()}")
print(f"  Value counts:")
for val, cnt in df["F3912"].value_counts().items():
    print(f"    {val}: {cnt:,}")

match = (df["F3912"] == df["F3924"]).sum()
total = len(df)
mismatch = total - match

print(f"\n  Exact match with F3924: {match:,} / {total:,}  ({match / total * 100:.2f}%)")
print(f"  Mismatches:             {mismatch}")

print(f"\n📊 Cross-tabulation (F3912 × F3924):")
ct = pd.crosstab(df["F3912"], df["F3924"], margins=True, margins_name="Total")
print(ct.to_string())

print()
print("📋 EVIDENCE:")
print("  1. F3912 has only 2 unique values: 0 and 1 — identical structure to target.")
print(f"  2. {match / total * 100:.2f}% of values are identical to F3924.")
print(f"  3. Only {mismatch} mismatches in {total:,} records.")
print("  4. The crosstab shows near-perfect diagonal alignment.")
print("  5. This is NOT a useful feature — it IS the answer.")
print()
print("⚖️  VERDICT:  CONFIRMED TARGET LEAKAGE.")
print("🚨 DECISION: MUST REMOVE in Phase 2 — non-negotiable.")
print()
print("   Including this feature would give any model near-perfect accuracy")
print("   that is completely fraudulent and useless in production.")

🔍 FORENSIC INVESTIGATION: F3912

  Unique values: 2
  Value counts:
    0: 9,000
    1: 82

  Exact match with F3924: 9,077 / 9,082  (99.94%)
  Mismatches:             5

📊 Cross-tabulation (F3912 × F3924):
F3924     0   1  Total
F3912                 
0      8998   2   9000
1         3  79     82
Total  9001  81   9082

📋 EVIDENCE:
  1. F3912 has only 2 unique values: 0 and 1 — identical structure to target.
  2. 99.94% of values are identical to F3924.
  3. Only 5 mismatches in 9,082 records.
  4. The crosstab shows near-perfect diagonal alignment.
  5. This is NOT a useful feature — it IS the answer.

⚖️  VERDICT:  CONFIRMED TARGET LEAKAGE.
🚨 DECISION: MUST REMOVE in Phase 2 — non-negotiable.

   Including this feature would give any model near-perfect accuracy
   that is completely fraudulent and useless in production.


#### Forensic Summary — `F3912`

| Evidence | Finding |
|----------|---------|
| Unique values | 2 (same as target) |
| Value distribution | 0: 9,000 / 1: 82 (vs target 0: 9,001 / 1: 81) |
| Match rate with F3924 | **99.94 %** |
| Mismatches | Only **5** out of 9,082 |

**Crosstab Analysis:**

|  | F3924=0 | F3924=1 |
|--|---------|---------|
| **F3912=0** | 8,998 | 2 |
| **F3912=1** | 3 | 79 |

The near-perfect diagonal shows F3912 is essentially a **pre-computed fraud label**.
It was likely generated from the same labelling process that created F3924.

**Risk:** 🔴 **CRITICAL** — Any model including F3912 will achieve ~99.9% accuracy
by simply copying this column. This accuracy is meaningless and would mask the model's
true inability to detect fraud from behavioural signals.

**Decision:** ❌ **MUST REMOVE** — confirmed target leakage, non-negotiable.

## Section 12 — Business Feature Discovery

Among the ~4,000 anonymised features, a handful have interpretable categorical values
that reveal their business meaning. These are **gold** for domain-driven feature engineering.

| Feature | Likely Represents | Categories | Missing |
|---------|-------------------|------------|---------|
| `F3886` | **Account Type** | 17 (Savings, Current, MSME Micro, …) | 0 |
| `F3889` | **Recency Band** | 7 (G365D, L365D, L7D, L180D, …) | 0 |
| `F3890` | **Location Type** | 4 (M=Metro, SU=Semi-Urban, R=Rural, U=Urban) | 0 |
| `F3891` | **Occupation** | 7 (selfemployed, salaried, student, …) | 0 |
| `F3892` | **Gender** | 3 (M, F, O) | **2,598 (28.6 %)** |
| `F3893` | **Segment** | 2 (RETAIL, CORPORATE) | 0 |
| `F3888` | **Account Opening Date** | 4,292 unique dates (1900–2025) | 0 |
| `F2230` | **Data Period / Cohort** | 4 (Oct25, Sep25, Nov25, Aug25) | 0 |

In [21]:
business_features = {
    "F3886": "Account Type",
    "F3889": "Recency Band",
    "F3890": "Location Type",
    "F3891": "Occupation",
    "F3892": "Gender",
    "F3893": "Segment",
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

palette = sns.color_palette("viridis", 20)

for i, (col, title) in enumerate(business_features.items()):
    ax = axes[i // 3, i % 3]
    vc = df[col].value_counts()
    bars = ax.bar(range(len(vc)), vc.values,
                  color=palette[:len(vc)], edgecolor="black", linewidth=0.5)
    ax.set_title(f"{title}  ({col})", fontsize=13, fontweight="bold")
    ax.set_ylabel("Count", fontsize=11)
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(vc.index, rotation=45, ha="right", fontsize=9)
    # annotate top bar
    for j, v in enumerate(vc.values):
        if j < 3:
            ax.text(j, v + 40, f"{v:,}", ha="center", fontsize=8, fontweight="bold")

plt.suptitle("Business Feature Distributions", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/figures/business_features.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: reports/figures/business_features.png")

✅ Saved: reports/figures/business_features.png


In [22]:
# Investigate F3888 (Account Opening Date)
dates = pd.to_datetime(df["F3888"], format="mixed", errors="coerce")
print(f"F3888 — Account Opening Date")
print(f"  Earliest: {dates.min()}")
print(f"  Latest:   {dates.max()}")
print(f"  Null after parsing: {dates.isnull().sum()}")
print()

fig, ax = plt.subplots(figsize=(14, 4))
dates.dt.year.value_counts().sort_index().plot(kind="bar", ax=ax,
    color="#3498db", edgecolor="black", linewidth=0.3)
ax.set_title("Account Opening Year Distribution (F3888)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/figures/account_opening_year.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: reports/figures/account_opening_year.png")

F3888 — Account Opening Date
  Earliest: 1900-01-03 00:00:00
  Latest:   2025-11-28 00:00:00
  Null after parsing: 0



✅ Saved: reports/figures/account_opening_year.png


In [23]:
# Investigate F2230 (Data Period / Cohort)
print("F2230 — Data Period / Cohort")
print(df["F2230"].value_counts())
print()
print("This appears to be the reporting period / data extraction cohort.")
print("Most records (9,001) belong to 'Oct25', aligning with the non-fraud majority.")

F2230 — Data Period / Cohort
F2230
Oct25    9001
Sep25      48
Nov25      23
Dec25      10
Name: count, dtype: int64

This appears to be the reporting period / data extraction cohort.
Most records (9,001) belong to 'Oct25', aligning with the non-fraud majority.


### Business Feature Insights

1. **Account Type (F3886):** Dominated by Savings (65.6%) and Current (22.6%) accounts.
   MSME and specialised products are minority classes.

2. **Recency Band (F3889):** G365D (Greater than 365 Days) dominates at 83.1%.
   This likely means most accounts are mature. Recent accounts (L7D, L14D) are rare — and may
   correlate with higher fraud risk.

3. **Location (F3890):** Roughly balanced across Metro, Semi-Urban, Rural, and Urban.

4. **Occupation (F3891):** Self-employed is the largest group (43.5%), followed by salaried (21%).

5. **Gender (F3892):** ⚠️ **28.6% missing** — this needs imputation or a separate "Unknown" category.
   The 61 "O" (Other) records are very few — risk of bias.

6. **Segment (F3893):** 70.9% RETAIL vs 29.1% CORPORATE.

7. **Date (F3888):** Spans 1900–2025. The 1900 dates are likely placeholder/default values — data quality issue.

8. **Period (F2230):** Data extraction cohort. Should not be used as a model feature (temporal leakage risk).

## Section 13 — Initial Observations

### ✅ Dataset Strengths

- **Rich engineered feature space** — 3,917 numeric features capturing diverse behavioural patterns
- **Multiple time-window indicators** — features span different recency bands (7D, 14D, 31D, 90D, 180D, 365D)
- **Business metadata available** — account type, occupation, location, segment provide interpretable context
- **Zero duplicate records** — every observation is unique
- **Key categorical features fully populated** — F3886, F3889, F3890, F3891, F3893 have zero missing values
- **Clean target variable** — F3924 is binary with no missing values

### ⚠️ Dataset Weaknesses

| Issue | Severity | Details |
|-------|----------|---------|
| **Severe class imbalance** | 🔴 Critical | Only 0.89% fraud (81/9,082) — 111:1 ratio |
| **Confirmed target leakage** | 🔴 Critical | F3912 is 99.94% identical to F3924 |
| **Index artefact leakage** | 🟠 High | `Unnamed: 0` carries positional correlation |
| **Extreme dimensionality** | 🟠 High | 3,925 features for 9,082 samples |
| **Massive missing values** | 🟠 High | 1,138 features have >50% missing data |
| **63 fully empty columns** | 🟡 Medium | No information at all |
| **Extreme skewness** | 🟡 Medium | Many features with \|skew\| > 50 |
| **Gender (F3892) bias risk** | 🟡 Medium | 28.6% missing + tiny "O" category |
| **Suspicious dates** | 🟡 Medium | F3888 has dates back to 1900 (likely defaults) |
| **Temporal cohort indicator** | 🟡 Medium | F2230 should not be a model feature |

## Section 14 — Decisions for Phase 2

These decisions form the basis of the data-cleaning pipeline in Phase 2.

| # | Decision | Reason |
|---|----------|--------|
| 1 | Remove 63 fully empty columns | No information whatsoever |
| 2 | Remove columns with >95% missing (361 cols) | Excessive sparsity — unreliable for modelling |
| 3 | Remove `Unnamed: 0` | Row-index artefact — positional leakage risk |
| 4 | **Remove `F3912`** | **Confirmed target leakage** (99.94% match with F3924) |
| 5 | Investigate columns with >50% missing | May contain recoverable signal for specific account types |
| 6 | Keep all categorical columns | Business information critical for AML interpretability |
| 7 | Parse `F3888` as datetime | Extract temporal features (account age, seasonality) |
| 8 | Handle `F3892` missing values | 28.6% missing — impute as "Unknown" category |
| 9 | Investigate `F2230` | Data cohort indicator — likely should not be a model feature |
| 10 | Address class imbalance | Use SMOTE / ADASYN / cost-sensitive learning in modelling phase |
| 11 | Evaluate high-skewness features | May need log/Box-Cox transformation or robust scaling |

## Section 15 — Save Audit Report

Export the audit summary as a structured CSV for downstream consumption and documentation.

In [24]:
audit_summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "fraud_cases": int(df["F3924"].sum()),
    "non_fraud_cases": int((df["F3924"] == 0).sum()),
    "fraud_percentage": round(df["F3924"].mean() * 100, 4),
    "imbalance_ratio": f"{int((df['F3924'] == 0).sum() / (df['F3924'] == 1).sum())}:1",
    "duplicate_rows": int(df.duplicated().sum()),
    "empty_columns": int((df.isnull().all()).sum()),
    "gt95_missing_columns": int(((df.isnull().sum() / len(df)) * 100 > 95).sum()),
    "gt50_missing_columns": int(((df.isnull().sum() / len(df)) * 100 > 50).sum()),
    "zero_missing_columns": int(((df.isnull().sum() / len(df)) * 100 == 0).sum()),
    "numeric_features": len(df.select_dtypes(include="number").columns),
    "categorical_features": len(df.select_dtypes(include=["object", "string"]).columns),
    "confirmed_leakage_features": "F3912, Unnamed: 0",
    "memory_usage_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
}

summary_series = pd.Series(audit_summary, name="value")
summary_series.to_csv("../reports/tables/data_audit_summary.csv")

print("✅ Audit summary saved to: reports/tables/data_audit_summary.csv\n")
for k, v in audit_summary.items():
    print(f"  {k:.<35s} {v}")

✅ Audit summary saved to: reports/tables/data_audit_summary.csv

  rows............................... 9082
  columns............................ 3925
  fraud_cases........................ 81
  non_fraud_cases.................... 9001
  fraud_percentage................... 0.8919
  imbalance_ratio.................... 111:1
  duplicate_rows..................... 0
  empty_columns...................... 63
  gt95_missing_columns............... 361
  gt50_missing_columns............... 1138
  zero_missing_columns............... 90
  numeric_features................... 3917
  categorical_features............... 8
  confirmed_leakage_features......... F3912, Unnamed: 0
  memory_usage_mb.................... 288.52


---

## ✅ Phase 1 Checklist

- [x] Notebook structure — 15 sections, clear narrative flow
- [x] Dataset overview — 9,082 × 3,925, 288 MB
- [x] Class imbalance analysis — 0.89% fraud, 111:1 ratio
- [x] Missing value analysis — 63 empty, 361 >95%, 1,138 >50%
- [x] Duplicate analysis — zero duplicates
- [x] Data type audit — 3,876 float, 41 int, 8 categorical
- [x] Categorical audit — 8 columns profiled
- [x] Numerical audit — skewness profiled, many |skew| > 50
- [x] Leakage investigation — F3912 confirmed, Unnamed: 0 flagged
- [x] Business feature discovery — 7 interpretable features documented
- [x] Initial observations — strengths, weaknesses, risks
- [x] Decisions documented — 11 actions for Phase 2
- [x] Audit summary exported — `reports/tables/data_audit_summary.csv`
- [x] All figures saved — `reports/figures/`

---

**Phase 1 Complete.** ✅

The dataset has been thoroughly audited. We have a clear understanding of its structure,
quality, risks, and opportunities. We are ready to proceed to **Phase 2 — Data Cleaning**.